# 08 — Admissibility-aware recourse baselines (revision)

Answers the "weak comparison" review point by testing the injection-point
result against two **independent** admissibility-aware recourse generators,
not just three handlings of one DiCE pipeline.

- **B1 soft-penalty**: constraints in the *objective* (actionable-recourse style)
- **B2 Monte-Carlo**: optimiser-free rejection sampling *inside* the region

Both reuse `guardrail_core` so the admissible region is identical to C1/C2.
See `baseline_core.py`. This notebook runs the cohort and the lambda sweep and
prints the headline numbers used in Table 3 and Table B2 of the paper.

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import subprocess, sys
# Run the two cohort experiments (cached to results/tables/)
print("Running baselines on 155-patient cohort ...")
subprocess.run([sys.executable, "run_baselines.py"], check=True)

In [ ]:
# Lambda sensitivity sweep for B1 (pre-empts the under-weighted-penalty objection)
print("Running B1 lambda sweep ...")
subprocess.run([sys.executable, "run_b1_lambda_sweep.py"], check=True)

In [ ]:
import pandas as pd, numpy as np
base = pd.read_csv("../results/tables/baselines_cohort.csv")
def wilson(k,n,z=1.96):
    if n==0: return (0,0)
    p=k/n; d=1+z*z/n; c=(p+z*z/(2*n))/d
    h=z*np.sqrt(p*(1-p)/n+z*z/(4*n*n))/d
    return (max(0,c-h)*100, min(1,c+h)*100)
for cond in ["B1_soft","B2_mc"]:
    s=base[base["cond"]==cond]; k=int(s["feasible"].sum()); n=len(s)
    lo,hi=wilson(k,n)
    print(f"{cond}: feasibility {k/n*100:.1f}%  95%CI[{lo:.1f},{hi:.1f}]  (n={n})")
b1=base[base["cond"]=="B1_soft"]
print(f"B1 reached target tier: {b1['reached_target'].mean()*100:.1f}%")
print(f"B1 reached but outside region: {b1['reached_but_outside'].mean()*100:.1f}%")

In [ ]:
# Lambda sweep table
print(pd.read_csv("../results/tables/b1_lambda_sweep.csv").to_string(index=False))